# DART 공시 분류기 파인튜닝

- **모델**: `klue/bert-base`
- **입력**: 공시 본문 텍스트 (1,500자)
- **출력**: 사업보고서 / 유상증자 / 감사보고서
- **데이터**: 979개 (사업보고서 355 / 감사보고서 316 / 유상증자 308)

## 1. 환경 확인

In [1]:
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
print(f'GPU: {torch.cuda.get_device_name(0)}')

PyTorch: 2.6.0+cu124
CUDA: True
GPU: NVIDIA GeForce RTX 4070 Ti


## 2. 데이터 로드 및 확인

In [3]:
import pandas as pd
import numpy as np

df = pd.read_csv('./data/dart_corpus_text.csv', encoding='utf-8-sig')
print(f'전체 샘플: {len(df)}')
print()
print('카테고리별 분포:')
print(df['label'].value_counts())
print()
print('텍스트 길이 통계:')
print(df['text'].str.len().describe())

전체 샘플: 979

카테고리별 분포:
label
사업보고서    355
감사보고서    316
유상증자     308
Name: count, dtype: int64

텍스트 길이 통계:
count     979.000000
mean     1316.246170
std       351.452207
min       107.000000
25%      1425.000000
50%      1500.000000
75%      1500.000000
max      1500.000000
Name: text, dtype: float64


In [4]:
# 레이블 인코딩
label_names = ['감사보고서', '사업보고서', '유상증자']
label2id = {l: i for i, l in enumerate(label_names)}
id2label = {i: l for i, l in enumerate(label_names)}

df['label_id'] = df['label'].map(label2id)
print('레이블 매핑:', label2id)
print(df[['label', 'label_id']].drop_duplicates())

레이블 매핑: {'감사보고서': 0, '사업보고서': 1, '유상증자': 2}
     label  label_id
0    사업보고서         1
355   유상증자         2
663  감사보고서         0


## 3. Train / Validation 분리

In [5]:
from sklearn.model_selection import train_test_split

train_df, val_df = train_test_split(
    df, test_size=0.2, random_state=42, stratify=df['label_id']
)
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

print(f'Train: {len(train_df)}개')
print(f'Val:   {len(val_df)}개')
print()
print('Train 분포:')
print(train_df['label'].value_counts())
print()
print('Val 분포:')
print(val_df['label'].value_counts())

Train: 783개
Val:   196개

Train 분포:
label
사업보고서    284
감사보고서    253
유상증자     246
Name: count, dtype: int64

Val 분포:
label
사업보고서    71
감사보고서    63
유상증자     62
Name: count, dtype: int64


## 4. 토크나이저 및 Dataset 구성

In [6]:
from transformers import AutoTokenizer
from torch.utils.data import Dataset

MODEL_NAME = 'klue/bert-base'
MAX_LENGTH = 256  # 공시 본문은 길어서 256 사용

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print(f'토크나이저 로드 완료: {MODEL_NAME}')

# 텍스트 길이 확인
sample_tokens = tokenizer(df['text'].iloc[0], truncation=True, max_length=MAX_LENGTH)
print(f'샘플 토큰 수: {len(sample_tokens["input_ids"])}')

토크나이저 로드 완료: klue/bert-base
샘플 토큰 수: 256


In [7]:
class DartDataset(Dataset):
    def __init__(self, df, tokenizer, max_length):
        self.texts = df['text'].tolist()
        self.labels = df['label_id'].tolist()
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            padding='max_length',
            truncation=True,
            max_length=self.max_length,
            return_tensors='pt'
        )
        return {
            'input_ids':      encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'labels':         torch.tensor(self.labels[idx], dtype=torch.long)
        }

train_dataset = DartDataset(train_df, tokenizer, MAX_LENGTH)
val_dataset   = DartDataset(val_df,   tokenizer, MAX_LENGTH)

print(f'Train Dataset: {len(train_dataset)}개')
print(f'Val Dataset:   {len(val_dataset)}개')

Train Dataset: 783개
Val Dataset:   196개


## 5. 모델 로드 및 학습 설정

In [8]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.metrics import accuracy_score, f1_score, classification_report

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3,
    id2label=id2label,
    label2id=label2id
)

total_params = sum(p.numel() for p in model.parameters())
print(f'총 파라미터: {total_params:,}')

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        'accuracy': accuracy_score(labels, preds),
        'f1_macro': f1_score(labels, preds, average='macro')
    }

training_args = TrainingArguments(
    output_dir='./models/dart_classifier_output',
    num_train_epochs=5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=3e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1_macro',
    fp16=True,
    logging_steps=20,
    report_to='none'
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)

print('학습 설정 완료')

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: klue/bert-base
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on you

총 파라미터: 110,619,651
학습 설정 완료


## 6. 학습

In [9]:
print('학습 시작...')
print(f'Train: {len(train_dataset)}개 | Val: {len(val_dataset)}개')
print(f'배치 크기: {training_args.per_device_train_batch_size} | 에폭: {training_args.num_train_epochs}')
trainer.train()

학습 시작...
Train: 783개 | Val: 196개
배치 크기: 16 | 에폭: 5


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,0.141308,0.004210,1.000000,1.000000
2,0.033418,0.001247,1.000000,1.000000
3,0.007501,0.001075,1.000000,1.000000
4,0.017093,0.000884,1.000000,1.000000
5,0.001142,0.000753,1.000000,1.000000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

TrainOutput(global_step=245, training_loss=0.0977901889505435, metrics={'train_runtime': 36.1744, 'train_samples_per_second': 108.226, 'train_steps_per_second': 6.773, 'total_flos': 515044515202560.0, 'train_loss': 0.0977901889505435, 'epoch': 5.0})

## 7. 평가

In [10]:
results = trainer.evaluate()
print('=== 최종 평가 결과 ===')
print(f"Accuracy : {results['eval_accuracy']:.4f}")
print(f"F1 Macro : {results['eval_f1_macro']:.4f}")

=== 최종 평가 결과 ===
Accuracy : 1.0000
F1 Macro : 1.0000


In [11]:
# 카테고리별 상세 리포트
pred_output = trainer.predict(val_dataset)
preds = np.argmax(pred_output.predictions, axis=-1)
labels_true = pred_output.label_ids

print(classification_report(labels_true, preds, target_names=label_names))

              precision    recall  f1-score   support

       감사보고서       1.00      1.00      1.00        63
       사업보고서       1.00      1.00      1.00        71
        유상증자       1.00      1.00      1.00        62

    accuracy                           1.00       196
   macro avg       1.00      1.00      1.00       196
weighted avg       1.00      1.00      1.00       196



## 8. 추론 테스트

In [12]:
from transformers import pipeline

classifier = pipeline(
    'text-classification',
    model=model,
    tokenizer=tokenizer,
    device=0
)

# 실제 검증 데이터로 테스트
print('=== 추론 테스트 (검증 데이터 샘플) ===')
for label in label_names:
    sample = val_df[val_df['label'] == label].iloc[0]
    result = classifier(sample['text'][:512])[0]
    print(f'\n실제: {label}')
    print(f'예측: {result["label"]} (confidence: {result["score"]:.3f})')
    print(f'텍스트: {sample["text"][:100]}...')

=== 추론 테스트 (검증 데이터 샘플) ===

실제: 감사보고서
예측: 감사보고서 (confidence: 0.993)
텍스트: 감사보고서 4.1 주식회사 미르디앤씨 01135765 411000000000 - 1801111149459 000 0 0 0 0 Y N N N J E N 4 N 주식회사 미르디앤씨 ...

실제: 사업보고서
예측: 사업보고서 (confidence: 0.996)
텍스트: 사업보고서 5.2 주식회사체시스 C A 사 업 보 고 서 (제 34 기) 목 차 【 대표이사 등의 확인 】 우리는 당사의 대표이사 및 신고업무담당이사로서 이 공시서류의 기재내용에 ...

실제: 유상증자
예측: 유상증자 (confidence: 0.996)
텍스트: 주요사항보고서(유상증자결정) 2.9 주식회사 맥스트 주요사항보고서 / 거래소 신고의무 사항 유상증자 결정 ※ 기타주식에 관한 사항 20. 기타 투자판단에 참고할 사항 가. 발행가액...


## 9. 모델 저장

In [13]:
SAVE_DIR = './models/dart_classifier'
model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print(f'모델 저장 완료: {SAVE_DIR}')

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

모델 저장 완료: ./models/dart_classifier


In [ ]:
from transformers import pipeline

classifier = pipeline('text-classification', 
                       model='./models/dart_classifier', 
                       tokenizer='./models/dart_classifier', device=0)

# test
test_text = """. 자금조달의 목적	시설자금(원)	-
영업양수자금(원)	-
운영자금(원)	-
채무상환자금(원)	72,000,000,000
타법인 증권 취득자금(원)	-
기타자금(원)	-"""
print(classifier(test_text[:512]))

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[{'label': '유상증자', 'score': 0.8360995054244995}]
